# Happiest Health: load and explore the structured CSVs
See `DATA_OVERVIEW.md` for what every file and column means.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
DATA = Path("../data") if Path("../data").exists() else Path("data")
pd.set_option("display.max_columns", 50)

# (file, datetime columns)
FILES = {
    "patients": ["dod"],
    "admissions": ["admittime", "dischtime", "deathtime"],
    "diagnoses": [],
    "labs": ["charttime"],
    "meds": ["starttime", "stoptime"],
    "vitals_hourly": ["hour"],
    "rhythm": ["charttime"],
    "ecg": ["ecg_time"],
    "events": ["event_time"],
    "wearable_5min": ["time"],
    "icare_patients": [],
}
dfs = {name: pd.read_csv(DATA / f"{name}.csv", parse_dates=dates, low_memory=False) for name, dates in FILES.items()}
for c in ("hadm_id",):  # nullable integer ids
    for name in ("labs", "ecg"):
        dfs[name][c] = dfs[name][c].astype("Int64")
globals().update(dfs)  # patients, admissions, ... as variables
pd.DataFrame({n: d.shape for n, d in dfs.items()}, index=["rows", "cols"]).T

## Column profile of every file
dtype, % missing, unique values, and range / top values.

In [ ]:
def profile(df):
    out = []
    for c in df.columns:
        s = df[c]
        row = {"column": c, "dtype": str(s.dtype), "missing_%": round(s.isna().mean() * 100, 1), "unique": s.nunique()}
        if pd.api.types.is_numeric_dtype(s) and s.nunique() > 2:
            row.update(min=s.min(), median=s.median(), max=s.max())
        else:
            row["top"] = str(s.value_counts().head(4).to_dict())[:80]
        out.append(row)
    return pd.DataFrame(out)

for name, df in dfs.items():
    print(f"\n===== {name}  {df.shape}")
    display(profile(df))

## Class balance of the candidate labels

In [ ]:
print("in-hospital death per admission:", admissions.hospital_expire_flag.value_counts().to_dict())
print("30-day readmission:", admissions.readmit_30d.value_counts().to_dict())
print("events:", events.event.value_counts().to_dict())
print("I-CARE poor outcome:", icare_patients.poor_outcome.value_counts().to_dict())

## Clean the vitals: implausible values to NaN

In [ ]:
RANGES = {"hr": (20, 250), "spo2": (50, 100), "rr": (4, 60), "sbp": (50, 250),
          "dbp": (20, 150), "map": (30, 200), "temp_c": (30, 43)}
vit = vitals_hourly.copy()
for col, (lo, hi) in RANGES.items():
    bad = ~vit[col].between(lo, hi) & vit[col].notna()
    print(f"{col:7s} set {bad.sum():3d} implausible values to NaN")
    vit.loc[bad, col] = np.nan

## Early-warning label: deterioration in the next 6 h
Deterioration = first vasopressor, lactate >= 4 or in-hospital death. Hours at or after the first event are dropped (leakage).

In [ ]:
HORIZON = 6
first_ev = (events[events.event.isin(["vasopressor_start", "lactate_ge_4", "death_in_hospital"])]
            .groupby("hadm_id").event_time.min())
vit["first_event"] = vit.hadm_id.map(first_ev)
hours_to_event = (vit.first_event - vit.hour).dt.total_seconds() / 3600
vit = vit[~(hours_to_event <= 0)].copy()          # drop hours after the event started
vit["label_6h"] = hours_to_event.loc[vit.index].between(0, HORIZON, inclusive="right").astype(int)
print(vit.label_6h.value_counts(), f"positive rate {vit.label_6h.mean():.2%}")

## Attach static features and the latest ECG

In [ ]:
static = patients[["subject_id", "gender", "anchor_age", "bmi", "ever_hf", "ever_af", "ever_cad", "ever_ckd", "ever_dm"]]
model_df = vit.merge(static, on="subject_id", how="left")
ecg_feats = ecg[ecg.leads_agree == 1][["subject_id", "ecg_time", "hr_bpm", "sdnn_ms", "rmssd_ms", "rr_cv"]]
model_df = pd.merge_asof(model_df.sort_values("hour"), ecg_feats.sort_values("ecg_time"),
                         left_on="hour", right_on="ecg_time", by="subject_id", direction="backward")
model_df.shape